# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

%pip install docling markdown-it-py
%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Document
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
with open(glob.glob(f"{data_dir}/*.md")[0], "r") as f:
    text = f.read()

In [ ]:
# Step 4: Text Chunking and Dataset Creation

from markdown_it import MarkdownIt
from typing import List
import datasets
import re


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    # This ensures we don't split in the middle of headings, lists, etc.
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # Emit a complete chunk
                chunks.append(" ".join(current_words))
                # Prepare next buffer with overlap from the end of this chunk
                # This ensures context continuity between chunks
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def build_icl(markdown_text: str, chunks: List[str]) -> dict:
    """
    Build ICL metadata dynamically from the processed markdown document.
    """
    heading_match = re.search(r"^#\s+(.+)$", markdown_text, flags=re.MULTILINE)
    if heading_match:
        document_outline = heading_match.group(1).strip()
    else:
        first_content_line = next((line.strip() for line in markdown_text.splitlines() if line.strip()), "")
        document_outline = first_content_line[:140] if first_content_line else "Document summary"

    domain_keywords = {
        "Finance": ["finance", "financial", "bank", "payment", "money", "tax", "fraud", "compliance", "aml", "kyc"],
        "Technology": ["software", "system", "api", "database", "model", "algorithm", "cloud", "code", "python", "ai"],
        "Healthcare": ["health", "patient", "clinical", "medical", "hospital", "treatment", "disease", "diagnosis"],
        "Legal": ["law", "legal", "regulation", "article", "section", "court", "contract", "rights"],
        "Education": ["education", "student", "teacher", "curriculum", "learning", "school", "assessment", "classroom"],
    }

    text_lower = markdown_text.lower()
    domain_scores = {
        domain: sum(text_lower.count(keyword) for keyword in keywords)
        for domain, keywords in domain_keywords.items()
    }
    best_domain = max(domain_scores, key=domain_scores.get)
    domain = best_domain if domain_scores[best_domain] > 0 else "General"

    icl_document = markdown_text.strip()[:3000]
    if not icl_document and chunks:
        icl_document = chunks[0][:3000]

    return {
        "document_outline": document_outline,
        "icl_document": icl_document,
        "icl_query_1": f"What is the main objective of '{document_outline}'?",
        "icl_query_2": "What are the most important points in the provided excerpt?",
        "icl_query_3": f"Which {domain.lower()} concepts, procedures, or requirements are described in this excerpt?",
        "domain": domain,
    }


chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)
if not chunks and text.strip():
    chunks = [text.strip()]


# Prepare seed data for the SDG-Hub knowledge pipeline.
#
# The seed data requires the following fields:
#   - document_outline: A concise title or summary that accurately represents the entire document.
#     For documents covering multiple themes, consider providing multiple outlines (one per section).
#   - icl_document: A representative sample extract from the document. This may include tables, code snippets, definitions, etc.
#   - icl_query_1, icl_query_2, icl_query_3: Three questions based on the icl_document sample.
#   - domain: The domain or subject area of the document.
#
# The code below creates a HuggingFace Dataset from the document chunks,
# then maps the required ICL fields to each entry, and finally saves the result as a JSONL file.

seed_data = datasets.Dataset.from_dict({"document": chunks})
icl = build_icl(text, chunks)

# Map the ICL fields to each document chunk (if you want to use the same ICL for all, as shown here)
seed_data = seed_data.map(lambda x: icl)

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)

### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook